[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/beer-sakthai/openenv-rl-training/blob/main/sakthai-agentic-eval-train/eval_sakthai_bench_colab.ipynb) [![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://www.kaggle.com/kernels/welcome?src=https://github.com/beer-sakthai/openenv-rl-training/blob/main/sakthai-agentic-eval-train/eval_sakthai_bench_colab.ipynb)

# SakThai bench-v2 — Corrected-Scorer & PEFT Adapter Evaluation (Free Colab T4)

Runs [`eval_bench_peft.py`](./scripts/eval_bench_peft.py) against `Nanthasit/sakthai-bench-v2` — byte-identical in rendering and scoring to the canonical `eval_bench.py`, upgraded with:
1. **Raw `data/test.jsonl` fallback** when `datasets.load_dataset` raises `ValueError: Feature type 'Json' not found` on Colab.
2. **Bare PEFT adapter support (`-tools`)** so both `-merged` models and LoRA adapters (`sakthai-context-{0.5b,1.5b,7b}-tools`) can be evaluated on the exact same harness.

**Runtime:** `Runtime ▸ Change runtime type ▸ T4 GPU`.

**One model per cell, on purpose.** Each invocation uploads its own `results/*.json` on completion, so a reclaimed Colab session never loses earlier finished models.

| Cell | Model | Weights (fp16) | Free T4 (~15.4 GB) |
|---|---|---|---|
| 4 | `sakthai-context-1.5b-merged` / `1.5b-tools` | ~3.1 GB | comfortable |
| 5 | `sakthai-context-0.5b-merged` / `0.5b-tools` | ~1.0 GB | comfortable |
| 6 | `sakthai-context-7b-merged` / `7b-tools` | ~15.2 GB | **skips on a T4** — needs ≥20 GB |

## 1. Environment + GPU check

Stops early with a clear message if no GPU is attached, rather than silently scoring on CPU for hours.

In [ ]:
!pip install -q "transformers>=4.44" datasets accelerate peft huggingface_hub

import torch

assert torch.cuda.is_available(), (
    "No GPU. Runtime > Change runtime type > T4 GPU, then re-run this cell."
)

NAME = torch.cuda.get_device_name(0)
TOTAL_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {NAME}  |  {TOTAL_GB:.1f} GB  |  bf16 supported: {torch.cuda.is_bf16_supported()}")
print("note: a T4 is Turing, so this run pins fp16 — bf16 loads there but dies in generate()")

## 2. Hugging Face login

Needs a **write** token so each run can upload its own results. Without it the eval still finishes and prints the payload between `BENCH_RESULTS_JSON_BEGIN/END` markers.

In [ ]:
from huggingface_hub import notebook_login, whoami

notebook_login()
print("logged in as:", whoami()["name"])

## 3. Fetch the PEFT-enabled corrected scorer

Fetches `eval_bench_peft.py` and `pipeline_common.py` from `beer-sakthai/openenv-rl-training` (which includes the `data/test.jsonl` fallback for `ValueError: Feature type 'Json' not found` and bare PEFT adapter support).

In [ ]:
import os
import subprocess

REPO = "Nanthasit/sakthai-bench-v2"
GH_RAW = "https://raw.githubusercontent.com/beer-sakthai/openenv-rl-training/main/sakthai-agentic-eval-train/scripts"

for fname in ("pipeline_common.py", "eval_bench_peft.py"):
    subprocess.run(["wget", "-q", "-O", fname, f"{GH_RAW}/{fname}"], check=True)

src = open("eval_bench_peft.py").read()
code = "\n".join(l for l in src.splitlines() if not l.lstrip().startswith("#"))
assert "selection_ok" in src, "fetched an invalid evaluator — expected corrected multiset scorer"
assert ".issubset(" not in code, "evaluator still scores parallel with set-subset"
assert "_load_bench_dataset" in src, "evaluator missing raw JSONL fallback for 'Json' feature error"
print(f"eval_bench_peft.py OK — {len(src.splitlines())} lines, PEFT + JSONL fallback + multiset scorer")

# Shared by every scoring cell below.
os.environ["SAK_BENCH"] = REPO
os.environ["SAK_UPLOAD_TO"] = REPO
os.environ["SAK_DUMP_ROWS"] = "1"   # keeps per-row lines, so a future rescore stays free
os.environ["SAK_DTYPE"] = "fp16"    # explicit: a T4 cannot run bf16
print("upload target:", REPO)

## 4. Score the 1.5B (`-merged` or `-tools`) ← start here

Roughly 10–20 min on a T4. Uploads its own `results/*.json` on completion, so this is banked before the next cell starts.

In [ ]:
os.environ["SAK_MODELS"] = "Nanthasit/sakthai-context-1.5b-merged"
os.environ["SAK_BATCH"] = "8"

!python eval_bench_peft.py 2>&1 | tee bench_1.5b.log | tail -30

## 5. Score the 0.5B (`-merged` or `-tools`)

The current best single-shot tool-caller — 91.0%–91.2% selection after rescoring. Roughly 5–10 min.

In [ ]:
os.environ["SAK_MODELS"] = "Nanthasit/sakthai-context-0.5b-merged"
os.environ["SAK_BATCH"] = "16"

!python eval_bench_peft.py 2>&1 | tee bench_0.5b.log | tail -30

## 6. Optional: 7B — only on a ≥20 GB GPU

Skips itself with an explanation on a free T4 instead of burning 20 minutes into an OOM. If Colab hands you an L4 (22 GB) or A100, it runs.

In [ ]:
if TOTAL_GB < 20:
    print(
        f"SKIPPING 7B: this GPU has {TOTAL_GB:.1f} GB; Qwen2.5-7B in fp16 needs "
        f"~15.2 GB of weights plus KV cache and will OOM.\n"
        f"Options: Runtime > Change runtime type > L4/A100, or score 7B on HF Jobs "
        f"once credits are topped up."
    )
else:
    os.environ["SAK_MODELS"] = "Nanthasit/sakthai-context-7b-tools"
    os.environ["SAK_BATCH"] = "4"
    !python eval_bench_peft.py 2>&1 | tee bench_7b.log | tail -30

## 7. Confirm what landed

The `scorer` column separates corrected runs (`multiset-selection-v2`) from the pre-fix era (`set-subset`).

In [ ]:
import json
from huggingface_hub import HfApi, hf_hub_download

api = HfApi()
subprocess.run(
    [
        "wget",
        "-q",
        "-O",
        "bench_history.py",
        f"https://huggingface.co/datasets/{REPO}/resolve/main/bench_history.py",
    ],
    check=True,
)
!python bench_history.py

todays = sorted(
    f
    for f in api.list_repo_files(REPO, repo_type="dataset")
    if f.startswith("results/") and f.endswith(".json")
)[-3:]
print("\n--- paste from here down ---")
for f in todays:
    d = json.load(open(hf_hub_download(REPO, f, repo_type="dataset")))
    if d.get("source"):
        continue  # derived payload, not a live run
    for r in d.get("results", []):
        if "error" in r:
            print(f"{f}  {r['model']}: ERROR {r['error'][:70]}")
            continue
        ho = r.get("held_out_tools") or {}
        held = f"{100 * ho['pass'] / ho['total']:.1f}" if ho.get("total") else "—"
        rt = r.get("runtime", {})
        print(
            f"{f}  {r['model'].split('/')[-1]:<30} "
            f"sel {r['overall_selection']:.1f}  args {r['overall_arguments']:.1f}  "
            f"strict {r['overall_strict']:.1f}  held {held}  "
            f"[{rt.get('dtype', '?')} on {rt.get('device', '?')}, scorer {d.get('scorer')}]"
        )
print("--- paste to here ---")